In [ ]:
from google.colab import drive
drive.mount('/content/drive')


# Fixed internal single-frame deletion diagnostic

Read the saved 1B M05 FULL181 once. Common source [2:180]; C removes tail 179 -> [2:179], D removes source 90 -> [2:90]+[91:180]. Both 177/R44; D also includes source 179 versus C. No generation/writer/codec.

Two framewise encodes (354 frames/46 batch8 calls), four independent key searches. Per read: 885 cached (r,d) cells, H0 b0..4 and H1 b0..3/k1..176, all 709 hypotheses, 225 local-grid cells. Original full-direction age normalization, including source 180 rho, is retained. Complete finite unique H0 and joint argmax use original tie_atol, no score gate. H1 has greater complexity; its victory is not detection.

RAW, GLOBAL_ALIGN and PATH_ALIGN are blind. PATH H1 inserts the previous received frame at estimated k, then prepends b%4 first frames and truncates to 177. This is a placeholder, not recovery of missing RGB. Use whole-clip Wan encoding, no truth-driven segmentation or VAE state reset; normal per-input cache clearing retained. Sync/plan/blind payload seals precede oracle-map parsing. TRUTH_PATH is oracle-only after blind seals. Message parsing occurs only after oracle payload seal.

Sixteen logical slots retain 675840 votes/22528 time-bit rows/512 final bits: 12 blind plus 4 oracle. Wan load 1, physical upper bounds 12 encodes/16 reads. Cache by full input index-map within observation; keys read separately. CPU normalized latents retain exact values and restore to the receiver device. Cached failures never retry. Oracle may reuse a blind same-map result, but never fills a blind unresolved slot.

Report family/b, map correct/177 and exact k errors separately from payload; C has no true k. Local received four-frame bins are display only, not Wan receptive fields. All failures/ties remain. No presence/FPR/generalization/scientific PASS or new attack scan. Agent only CPU/fake/static; user performs Run all.

Fixed input: /content/drive/MyDrive/Video-WM/Trajectory-Receiver-Independent-Source-Milestone1B-V1/20261006T110053146950Z/fixed_reference/source_preparation/M05_FRAMEWISE_SYNC/received.rgb8; SHA256 db63d12e3a767c78d93e25ebf420222d4d96cdb5e41c8b607f34a6f82b12a5bd

UNPUBLISHED DRAFT: SOURCE_SHA=None; publish reviewed source and bind its immutable SHA before Run all.

In [ ]:
SOURCE_SHA = None
FIXED = {'observations': 2, 'sync_reads': 4, 'frame_displacement_cells': 3540, 'path_candidates': 2836, 'local_grid_cells': 900, 'blind_logical_reads': 12, 'oracle_logical_reads': 4, 'logical_payload_reads': 16, 'logical_detailed_votes': 675840, 'logical_time_bit_rows': 22528, 'logical_final_bits': 512}
from pathlib import Path
import datetime,hashlib,json,os,signal,subprocess,sys,time,traceback
if SOURCE_SHA is None:
    raise RuntimeError('UNPUBLISHED_DRAFT: publish reviewed source and rebuild with its immutable SHA before Run all')
if not isinstance(SOURCE_SHA,str) or len(SOURCE_SHA)!=40:
    raise RuntimeError('immutable 40-character source SHA required')
STAMP=datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUTPUT=Path('/content/drive/MyDrive/Video-WM/Trajectory-Internal-Single-Deletion-V1')/STAMP
OUTPUT.mkdir(parents=True,exist_ok=False)
RUN_OUTPUT=OUTPUT/'fixed_reference'
REPO=Path('/content/SC-SSTW-TRAJECTORY-INTERNAL-SINGLE-DELETION-V1-'+STAMP)
PYTHON=sys.executable
def write_json(path,value):
    tmp=path.with_suffix(path.suffix+'.tmp');tmp.write_text(json.dumps(value,indent=2)+'\n');os.replace(tmp,path)
def failed(stage,exc):
    captured_traceback=''.join(traceback.format_exception(type(exc),exc,exc.__traceback__))
    write_json(OUTPUT/'setup_failure.json',dict(stage=stage,error=f'{type(exc).__name__}: {exc}',traceback=captured_traceback,fixed_denominator=FIXED))
def logged(command,stage,cwd=None,check=True):
    child=None;code=None;primary=None;primary_tb=None;cleanup_errors=[]
    def retain_cleanup_error(label,exc):
        nonlocal primary,primary_tb
        if primary is None and not isinstance(exc,Exception):
            primary=exc;primary_tb=exc.__traceback__
        else:
            cleanup_errors.append(label+': '+repr(exc))
    try:
        with (OUTPUT/'execution.log').open('a') as log:
            log.write('COMMAND '+repr(command)+'\n');log.flush()
            popen_group={'start_new_session':True} if os.name=='posix' else {}
            child=subprocess.Popen(command,cwd=cwd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,**popen_group)
            try:
                for line in child.stdout:
                    print(line,end='',flush=True);log.write(line);log.flush()
                code=child.wait()
            except BaseException as exc:
                primary=exc;primary_tb=exc.__traceback__
            finally:
                if child is not None:
                    if os.name=='posix':
                        pgid=child.pid
                        try:os.killpg(pgid,signal.SIGTERM)
                        except ProcessLookupError:pass
                        except BaseException as exc:retain_cleanup_error('group SIGTERM',exc)
                        try:
                            deadline=time.monotonic()+5.0
                            while time.monotonic()<deadline:
                                try:os.killpg(pgid,0)
                                except ProcessLookupError:break
                                except BaseException as exc:
                                    retain_cleanup_error('group probe',exc);break
                                time.sleep(0.05)
                        except BaseException as exc:
                            retain_cleanup_error('group TERM grace',exc)
                        finally:
                            try:os.killpg(pgid,signal.SIGKILL)
                            except ProcessLookupError:pass
                            except BaseException as exc:retain_cleanup_error('group SIGKILL',exc)
                    elif child.poll() is None:
                        try:child.terminate()
                        except ProcessLookupError:pass
                        except BaseException as exc:retain_cleanup_error('terminate',exc)
                    try:
                        try:code=child.wait(timeout=10)
                        except subprocess.TimeoutExpired:
                            try:child.kill()
                            except ProcessLookupError:pass
                            except BaseException as exc:retain_cleanup_error('kill',exc)
                            try:code=child.wait(timeout=10)
                            except BaseException as exc:retain_cleanup_error('wait after kill',exc)
                        except BaseException as exc:retain_cleanup_error('wait',exc)
                    finally:
                        if child.stdout is not None:
                            try:child.stdout.close()
                            except BaseException as exc:retain_cleanup_error('stdout close',exc)
            if primary is None and not cleanup_errors:
                log.write('EXIT '+str(code)+'\n');log.flush()
    except BaseException as wrapper_error:
        if primary is None:
            primary=wrapper_error;primary_tb=wrapper_error.__traceback__
        elif wrapper_error is not primary:
            cleanup_errors.append('log wrapper/close: '+repr(wrapper_error))
    if primary is None and cleanup_errors:
        primary=RuntimeError('process cleanup failed: '+'; '.join(cleanup_errors))
        primary_tb=primary.__traceback__
    if primary is None and check and code:
        primary=subprocess.CalledProcessError(code,command)
        primary_tb=primary.__traceback__
    if primary is not None:
        if cleanup_errors and hasattr(primary,'add_note'):
            primary.add_note('secondary cleanup errors: '+'; '.join(cleanup_errors))
        try:failed(stage,primary)
        except BaseException as record_error:
            if hasattr(primary,'add_note'):primary.add_note('failure record error: '+repr(record_error))
        raise primary.with_traceback(primary_tb)
    return code
write_json(OUTPUT/'setup_receipt.json',dict(status='SETUP_STARTED',source_sha=SOURCE_SHA,fixed_denominator=FIXED))


In [ ]:
try:
    logged(['git','clone','--filter=blob:none','https://github.com/RICHAAARC/SC-SSTW.git',str(REPO)],'SOURCE_CLONE')
    logged(['git','-C',str(REPO),'fetch','origin',SOURCE_SHA],'SOURCE_FETCH')
    logged(['git','-C',str(REPO),'checkout','--detach',SOURCE_SHA],'SOURCE_CHECKOUT')
    actual=subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
    dirty=subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True)
    if actual!=SOURCE_SHA or dirty:raise RuntimeError('source SHA/clean checkout mismatch')
    probe="import importlib.metadata as m; pins={'torch': '2.11.0', 'diffusers': '0.39.0', 'transformers': '4.57.6', 'numpy': '2.1.3', 'accelerate': '1.15.0', 'safetensors': '0.8.0', 'huggingface-hub': '0.36.2', 'tokenizers': '0.22.2'}; actual={k:m.version(k) for k in pins}; assert all(actual[k].split('+')[0]==v for k,v in pins.items()), actual; from diffusers import AutoencoderKL, AutoencoderKLWan; import torch; print(actual)"
    if logged([PYTHON,'-c',probe],'RECEIVER_DEPENDENCY_PROBE',check=False):
        logged([PYTHON,'-m','pip','install','torch==2.11.0','diffusers==0.39.0','transformers==4.57.6','numpy==2.1.3','accelerate==1.15.0','safetensors==0.8.0','huggingface-hub==0.36.2','tokenizers==0.22.2'],'RECEIVER_DEPENDENCY_REPAIR')
        logged([PYTHON,'-c',probe],'RECEIVER_DEPENDENCY_REPROBE')
    dependency_code=logged([PYTHON,'-m','pip','check'],'DEPENDENCY_REPORT',check=False)
    (OUTPUT/'environment_freeze.txt').write_text(subprocess.check_output([PYTHON,'-m','pip','freeze'],text=True))
    write_json(OUTPUT/'source_receipt.json',dict(source_sha=actual,clean=True))
    write_json(OUTPUT/'setup_receipt.json',dict(status='SETUP_COMPLETE',source_sha=actual,fixed_denominator=FIXED,dependency_check_returncode=dependency_code))
except Exception as exc:
    try:failed('SOURCE_OR_ENVIRONMENT',exc)
    except BaseException as record_error:
        if hasattr(exc,'add_note'):exc.add_note('failure record error: '+repr(record_error))
    raise


In [ ]:
try:
    from google.colab import userdata
    token=userdata.get('HF_TOKEN')
    if token:os.environ['HF_TOKEN']=token
except Exception:pass
finally:token=None
command=[PYTHON,'-u','-m','experiments.wan_state_clock.video_trajectory_internal_single_deletion_v1_run','--output',str(RUN_OUTPUT)]
try:
    returncode=logged(command,'FIXED_RUN',cwd=REPO,check=False)
    RESULT_PATH=RUN_OUTPUT/'result.json'
    write_json(OUTPUT/'execution_receipt.json',dict(command=command,returncode=returncode,result_path=str(RESULT_PATH),result_exists=RESULT_PATH.is_file()))
    if not RESULT_PATH.is_file():raise RuntimeError('No runner result; fixed denominator retained')
except Exception as exc:
    try:failed('RUNNER',exc)
    except BaseException as record_error:
        if hasattr(exc,'add_note'):exc.add_note('failure record error: '+repr(record_error))
    raise


In [ ]:
result=json.loads(RESULT_PATH.read_text())
if result['source_sha']!=SOURCE_SHA or result['fixed_denominator']!=FIXED:raise RuntimeError('result identity mismatch')
print('Status:',result['status'],'Counts:',result['counts'])
print('Calls:',result['calls'],'Stage calls:',result['stage_calls'])
print('Planned physical:',result['planned_physical_calls'],'Aliases are not independent evidence')
print('Environment:',result['environment'])
for key in ('blind_sync_seal','blind_plan_seal','blind_payload_seal','oracle_plan_seal','oracle_payload_seal'):
    print(key,result.get(key))
for sid,row in sorted(result['estimates'].items()):
    for family,estimate in row.items():
        print('Blind',sid,family,{k:estimate.get(k) for k in ('status','reason','path','best_score','gap')})
for sid,row in sorted(result['path_posthoc'].items()):
    for family in ('H0','joint'):
        e=row.get(family,{})
        print('Path posthoc',sid,family,'key=',row.get('key_role'),'view=',row.get('view'),
              {k:e.get(k) for k in ('family_correct','b_correct','map_correct','map_denominator','k_exact','k_signed_error','k_absolute_error','false_jump_on_this_input')})
for lid,row in sorted(result['payload_posthoc'].items()):
    margins=[x['signed_normalized_margin'] for x in row.get('bit_rows',[]) if 'signed_normalized_margin' in x]
    print(lid,row['status'],'key=',row.get('key_role'),'view=',row.get('view'),'oracle=',row.get('oracle'),
          'bit_errors=',row.get('bit_errors'),'error_bits=',row.get('error_bits'),'minmargin=',min(margins,default=None),'alias=',row.get('alias_of'))
    print('  detail:',result['payload_reads'][lid].get('detail_path'),row.get('path'))
print('Local grid/top evidence:',{sid:row['path'] for sid,row in result['sync_reads'].items()})
print('Failures:',result['failures'])
print('Ceiling:',result['evidence_ceiling'])
print('Result:',RESULT_PATH)
